# PyTorch Lightning fine-tuning template

by Andrés Muñoz-Jaramillo

This notebook is meant to act as a template to train and use a surya model to implement DS application.

It focuses on the concept of defining a modified Surya model, loading its weigths, and using a PyTorch lightning training loop to train it

This notebook assumes familiarity with the concepts of datasets and dataloaders contained in the **_0_dataset_dataloader_template.ipynb_**

It doesn't require having seen the baselines template, but they are meant to complement each other.  **_In fact they are on purpose almost identical!!!_**

## Set your cuda visible device

**IMPORTANT:** Since we are sharing resources, please make sure that the cuda visible device you put here is the one assigned to your team and your machine.   

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
# Must be set BEFORE torch is imported: cuBLAS reads this once, when it initializes, so
# setting it later has no effect. It is what lets training.deterministic work without a
# cuBLAS warning on every run. (Restart the kernel if torch was already imported.)
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import sys
from torch.utils.data import DataLoader

import torch
import yaml

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger, WandbLogger

# Append base path.  May need to be modified if the folder structure changes.
# It gives the notebook access to the wokshop_infrastructure folder.
sys.path.append("../../")
 
# Append Surya path. May need to be modified if the folder structure changes.
# It gives the notebook access to surya's release code.

from workshop_infrastructure.utils import build_scalers  # Data scaling utilities for Surya stacks
from workshop_infrastructure.utils import apply_peft_lora
torch.set_float32_matmul_precision('medium')



## Load configuration

Surya was designed to read a configuration file that defines many aspects of the model
including the data it uses we use this config file to set default values that do not
need to be modified, but also to define values specific to our downstream application

In [3]:
# The config is the single source of truth. load_wave_config() parses it into a typed
# object, exactly as the training script 4_finetune_wave_1D.py does, so the same YAML
# behaves identically here and in production. Notebook 0 walks through what it contains.
from downstream_apps.test.configs import load_wave_config

cfg = load_wave_config("./configs/config_script.yaml")
print(f"Loaded config for job: {cfg.job_id}")


Loaded config for job: wave_classification


## Download assets

The config says where the assets belong, so it is loaded first. `ensure_assets()` fetches only what is missing from HuggingFace, so re-running this is free.


In [4]:
# One implementation, shared by the notebooks, the training script and the
# download_*.sh wrappers: workshop_infrastructure/assets.py.
# Fine-tuning needs the pretrained backbone as well (~1.8 GB).
from workshop_infrastructure.assets import ensure_assets

ensure_assets(cfg, which=["scalers", "weights"])

# Now that scalers.yaml is guaranteed to be on disk, load it. build_scalers()
# accepts the resolved path directly.
scalers = build_scalers(info=cfg.data.scalers_path)
print(f"Loaded scalers for {len(scalers)} channels.")


Loaded scalers for 13 channels.


### Typed configuration, and how to extend it for your own task

`load_wave_config()` reads `configs/config_script.yaml` and returns a typed `TrainingConfig`.
This notebook and `4_finetune_wave_1D.py` call the same function on the same file, so
there is no notebook-versus-script divergence to reason about.

| YAML section | Access in Python | Dataclass |
|---|---|---|
| `data:` | `cfg.data.*` | `WaveDataConfig` (this app) |
| `model:` | `cfg.model.*` | `ModelConfig` |
| `model.lora_config:` | `cfg.model.lora_config.*` | `LoraAdapterConfig` |
| `model.time_embedding:` | `cfg.model.time_embedding.*` | `TimeEmbeddingConfig` |
| `training:` | `cfg.learning_rate`, `cfg.batch_size`, … | `TrainingConfig` (flat) |
| `output:` | `cfg.output.*` | `OutputConfig` |
| `logging:` | `cfg.wandb_project`, `cfg.wandb_entity` | `TrainingConfig` (flat) |

**Everything except `WaveDataConfig` lives in `workshop_infrastructure/configs.py`** and is
shared by every downstream app. When you fork the template you do not copy that file. You
subclass `DataConfig` with your task's fields and bind `load_config` to it — the whole of
`downstream_apps/template/configs.py` is:

```python
@dataclass
class WaveDataConfig(DataConfig):
    wave_index_path: str = ""
    ds_time_column: str = "start_time"
    ds_time_tolerance: str = "4d"
    ds_match_direction: str = "forward"
    ds_class_column: str = ""
    PATH_FIELDS = DataConfig.PATH_FIELDS + ("wave_index_path",)   # resolve it like a path

load_wave_config = partial(load_config, data_cls=WaveDataConfig)
```

Unknown keys are rejected rather than silently dropped: if you add a key to the YAML before
adding the field, you get an error naming the key and listing the valid ones.


## Define Downstream (DS) datasets

This child class takes as input all expected HelioFM parameters, plus additonal parameters relevant to the downstream application.  Here we focus in particular to the DS index and parameters necessary to combine it with the HelioFM index.

Another important component of creating a dataset class for your DS is turning the catalog's label column into something the loss can consume.  Here the catalog's `class` column holds the strings `"wave"` / `"no wave"`, and `label_transform` maps them to `1.0` / `0.0` for the binary cross-entropy loss.

Each catalog event appears **twice**: once as `wave` at `start_time + 30 min` and once as `no wave` at `start_time - 30 min`.  The two rows share a `start_time`, so they are a matched pair of the same event an hour apart — which is what makes the classification a controlled comparison rather than a comparison of different active regions.

In this case we will define both a training and a validation dataset using the indices pointed at in the config

**_Important:  the cap on dataset size comes from `data.max_samples` in the config (10 while exploring), not from a number typed into this notebook.  Keep this in mind in case the database seems smaller than you expect — and note that the cap head-slices a frame sorted by time, so it gives you the EARLIEST samples, not a random sample._**


In [5]:
from downstream_apps.test.datasets.wave_dataset import waveDSDataset

In [6]:
# build_helio_dataloaders() constructs the train and validation datasets and wraps them
# in DataLoaders. It fills in every generic argument (channels, temporal sampling, S3
# access, worker settings) from the config — see notebook 0 for what that block looks
# like written out. Only the wave-specific arguments are passed here, which is exactly
# the list you replace when you fork the template.
#
# It also handles two details that are easy to get wrong by hand: the validation set gets
# phase="val" (no random channel masking or flips), and only the training loader shuffles.
#
# One thing it does NOT expose is prefetch_factor, and it sets drop_last=True for the
# validation loader too. Neither matters at max_samples: 10, but both do at full scale --
# see experiments/wave_common.py:build_wave_loader() for why and what to do instead.
from workshop_infrastructure.datasets.builders import build_helio_dataloaders
import numpy as np
train_data_loader, val_data_loader = build_helio_dataloaders(
    cfg,
    waveDSDataset,
    scalers=scalers,
    num_workers=4,          # fewer workers than the script: notebooks start faster
    #### Downstream (DS) specific parameters
    return_surya_stack=True,
    # From the config, not a literal: data.max_samples is the one knob for dataset size,
    # so the notebook and 4_finetune_wave_1D.py cannot disagree about how much data ran.
    max_number_of_samples=cfg.data.max_samples,
    ds_wave_index_path=cfg.data.wave_index_path,
    ds_time_column=cfg.data.ds_time_column,
    ds_class_column=cfg.data.ds_class_column,
    ds_time_tolerance=cfg.data.ds_time_tolerance,
    ds_match_direction=cfg.data.ds_match_direction,
    label_transform=lambda s: (s == "wave").astype(np.float32),
)

batch_size = cfg.batch_size
print(f"train: {len(train_data_loader.dataset)} samples | "
      f"val: {len(val_data_loader.dataset)} samples | batch_size: {batch_size}")


train: 10 samples | val: 10 samples | batch_size: 2


Training and validation get separate datasets and dataloaders. They differ only in the index they read and in `phase`: `phase="val"` turns off the random channel masking and vertical flips used for training augmentation.

The loaders use `multiprocessing_context="spawn"` — the dataset holds an S3 client that does not survive `fork`, and spawn also avoids lockups in shared environments.


In [7]:
# Inspect a single batch to confirm shapes before building the model.
batch = next(iter(train_data_loader))
print({k: (tuple(v.shape) if hasattr(v, "shape") else type(v).__name__) for k, v in batch.items()})


{'ts': (2, 13, 1, 4096, 4096), 'time_delta_input': (2, 1), 'forecast': (2,), 'ds_index': 'list'}


## Initialize the HelioSpectformer model

This is the main difference beteween the notebook that trains the simple model and the one that fine-tunes Surya.  

In the case of the finetuning exercise one of the main differences between DS applications is the dimensionality of the output.  In this notebook we use a modified HelioSpectformer that projects into a 1D space. 

**_IMPORTANT: If your DS application is 2D you need to use the HelioSpectformer2D_**

In [8]:
from workshop_infrastructure.models.finetune_models import HelioSpectformer1D

/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Now the config file really comes into bear. The Spectformer has a metric ton of hyperparameters

In [9]:
# HelioSpectformer1D has a long list of architecture arguments, and all of them come
# straight from the model: section of the config. from_config() does that mapping, so the
# backbone can never drift out of sync with the checkpoint it is about to load.
#
# Arguments that are not part of ModelConfig (dtype, and anything from the training:
# section) are passed as explicit overrides.
model = HelioSpectformer1D.from_config(
    cfg.model,
    num_outputs=1,
    dtype=cfg.dtype,
    use_latitude_in_learned_flow=cfg.use_latitude_in_learned_flow,
)


## Load model weights

Here we load the pre-trained checkpoint and load the weights.  The exercise of loading follows the idea of us as many of the weights as possible.  This is accomplished through the filtered_checkpoint_state.   It checks to see if the pretrained model's layers match those of your finetuning architecture.   It also checks that all your dimensions across layers check out.   If something does not work those paramameters are left in their random initialization. 

In [10]:
# The checkpoint was saved from HelioSpectFormer directly, so its keys are flat
# (e.g. "embedding.proj.weight"), while the fine-tuning model nests the backbone under
# "backbone.*". load_pretrained_weights() tries both spellings and reports how many
# tensors matched — a low count means the architecture does not match the checkpoint.
from workshop_infrastructure.utils import load_pretrained_weights

load_pretrained_weights(model, cfg.model.pretrained_path)


Loading pretrained weights from /home/jovyan/surya_workshop2026/downstream_apps/test/assets/surya.366m.v1.pt.


Loaded 156 / 159 pretrained weights.


## To LoRA or not to Lora

This cell gives you two options.  On the one hand we have the classic freezing of the backbone (the initial layers of the model).   On the other hand we have the use of a LoRA.

LoRas have been a remarkable addition to our arsenal of models.   They have the advantage of keeping pretty much the entire model intact and only add broad modifications to weights as needed.

**What actually trains.** In the LoRA regime it is the adapters *and* the whole fine-tuning head. That second part is easy to get wrong: PEFT freezes every parameter it does not recognise as an adapter, so unless the head is explicitly handed to it as `modules_to_save`, the adapters end up fitting a **frozen, randomly initialised readout** — and the loss still goes down, so the training curve looks perfectly healthy. `apply_peft_lora()` avoids this by discovering every `head_*` attribute on the model and marking it trainable. If you add your own head layer, give it a `head_` prefix or it will be silently frozen (the helper checks this at startup and tells you what to rename).

**Where the adapters go.** `fc1`/`fc2` in all ten blocks, plus `attn.qkv` and `attn.proj` in the eight attention blocks. The spectral `complex_weight`, `attn.to_dynamic_projection`, and the patch embedding are never adapted.

Surya fuses query, key and value into a single `nn.Linear(1280, 3840)`, so one adapter covers all three at once: they share the `8×1280` matrix `A` and each gets its own `1280×8` slice of `B`. Their combined rank is at most 8 — which is *not* the same as giving q, k and v three independent rank-8 adapters.

Run the cell below and check the printout: the LoRA regime should report **3,157,761** trainable parameters (1,515,520 of adapters + 1,642,241 of head), and the linear probe **1,642,241**.

In [11]:
# Three fine-tuning regimes, all selected from the model: section of the config:
#
#   use_lora: true                          -> LoRA adapters + the whole head (default)
#   use_lora: false, freeze_backbone: true  -> linear probe: only the head trains
#   use_lora: false, freeze_backbone: false -> full fine-tuning of all 366M parameters
#
# freeze_backbone is ignored when use_lora is true: PEFT freezes everything, then
# re-enables the adapters and every head_* module.
#
# 3_finetune_template_1D.py applies exactly this logic in build_model().
if cfg.model.freeze_backbone:
    for name, param in model.named_parameters():
        if name.startswith("backbone."):
            param.requires_grad = False

if cfg.model.use_lora:
    # Prints the adapted modules and the trainable head modules it discovered.
    model = apply_peft_lora(model, cfg.model.lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

Applying PEFT LoRA: r=8, alpha=8, dropout=0.1, modules=['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2']


[LoRA] Adapted modules (20):
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.0.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.1.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.2.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.3.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.4.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.4.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.5.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.5.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.6.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.6.mlp.fc2
[LoRA]   backbone.backbone.blocks_attention.7.mlp.fc1
[LoRA]   backbone.backbone.blocks_attention.7.mlp.fc2
[LoRA]   backbone.backbone.blocks_spectral_gating.0.mlp.fc1
[LoRA]   backbone.backbone.blocks_spectral_gati

We can now test that this model manipulates a batch as expected and returns one number per sample, as we did for the simple baseline.

We pass the input stack `ts` to the model to get our classification output.  Note that the number returned is a **logit**, not a probability: `BCEWithLogitsLoss` applies the sigmoid itself, which is numerically better behaved than applying it in the model and taking a log afterwards.  Since the head is randomly initialized and the backbone was pretrained on a different task, these numbers mean nothing yet — this only tests that the forward pass has no dimension problems.

Dimension problemns are the dominant source of error in this kind of work.

Note that our output has now the size of our batch.


In [12]:
batch = next(iter(train_data_loader))
output = model.forward(batch)  # Get rid of singleton dimension
output

tensor([-0.0142, -0.3195], grad_fn=<SqueezeBackward1>)

## Define your metrics

Metrics are a very important part of training AI models.   They provide your models with the quantitification of error, which in turn shifts the weights towards better pefrorming models.  They also provide a way for you to monitor performance, identify overfitting, and quantify value added. 

We now initialize the metrics class which allows you to control what metrics do you want to use as "loss" (i.e. the metrics that backpropagate through your model) and which ones for monitoring performance.  As with other components, this takes the form of a loaded module that can be later use in a training script

In [13]:
from downstream_apps.test.metrics.template_metrics import WaveMetrics

In [14]:
train_loss_metrics = WaveMetrics("train_loss")
# val_loss is the quantity logged as "val_loss" and used to pick the best checkpoint.
# It defaults to the same BCE-with-logits as train_loss — override WaveMetrics.val_loss
# to monitor something else.
val_loss_metrics = WaveMetrics("val_loss")
train_evaluation_metrics = WaveMetrics("train_metrics")
# Reported only: val_metrics do NOT influence checkpoint selection.
validation_evaluation_metrics = WaveMetrics("val_metrics")

Now they can be evaluated on our model's output and our ground truth.   First the loss that actually will backpropagate, in this case **binary cross-entropy with logits**.

A useful reference point: a model that predicts the base rate and nothing else scores `ln 2 = 0.693` on a balanced set.  Anything at or above that has learned nothing about the input.


In [15]:
train_loss_metrics(output, batch["forecast"])

({'bce': tensor(0.6232, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)}, [1])

Then a training evaluation that will not backpropagate and inform our model, but that we can keep an eye on. Note that reporting lots of metrics during training will slow the training process.  I'm including it her as an example, but oftentimes is better to put the diagnostics only in the validation evaluation metrics.

Here we are calculating **binary accuracy** (https://lightning.ai/docs/torchmetrics/stable/classification/accuracy.html).  The event-level subsetting keeps the classes exactly balanced, so 0.5 is exactly chance and anything above it is signal.


In [16]:
train_evaluation_metrics(output, batch["forecast"])

({'accuracy': tensor(0.5000)}, [1])

In the validation evaluation metrics we report both accuracy and **AUROC**.

AUROC is worth watching alongside accuracy because it depends only on the *ranking* of the samples, not on where the 0.5 threshold falls.  That separation is diagnostic: a model whose accuracy drifts while its AUROC sits frozen has stopped changing its ranking and is only rescaling a direction it fixed early — which is what memorizing a handful of training samples looks like.


In [17]:
validation_evaluation_metrics(output, batch["forecast"])

({'accuracy': tensor(0.5000), 'auroc': tensor(1.)}, [1, 1])

## Define your PyTorch ligthning module

In this workshop we will use PyTorch lightning to train our models.  PyTorch lighting reduces the amount of code required to implement a training loop in comparison to PyTorch (at the expense of control and versatility).  

Opening the WaveLightningModule shows a simple Lightning model implementation.  It consists of:

- An initialization of the class (metrics, model, and learning rate).
- The forward code that runs evaluation of the model.
- Training and validation steps.
- Configuration of optimizers.

**_Note that it is the same Lightning module we used for the baseline!!_**

In [18]:
from downstream_apps.test.lightning_modules.pl_simple_baseline import WaveLightningModule

## Set your global seeds

Since training AI models generally uses stochastic gradient descent, it is a good idea to fix your random seeds so that your training exercise is reproducible.    

In [19]:
L.seed_everything(42, workers=True)

Seed set to 42


42

## Intialize Lightning module

Now we properly initalize the Lightning module to enable training, including passing the dictionary of metrics

In [20]:
metrics = {
    'train_loss': train_loss_metrics,
    'val_loss': val_loss_metrics,
    'train_metrics': train_evaluation_metrics,
    'val_metrics': validation_evaluation_metrics,
}

lit_model = WaveLightningModule(model, metrics, lr=cfg.learning_rate, batch_size=batch_size)


## Logging

In order to properly compare experiments against each other, it is very useful to log evaluation metrics in a place where they can be compared against other training runs.  In this workshop we will use Weights and Biases (WandB). 

The first time you run WandB in a machine it will ask you to login to WandB.  You should have received an invitation to our project.  In order to login you must:

- Select option 2 (existing account).   In VScode the dialog opens a box at the top of your screen.
- Click on get API Key (this will open a browser).
- Generate API Key.
- Paste it in the dialog box at the top of your VSCode

In [21]:
project_name = cfg.wandb_project
run_name = "finetune_experiment_1"  # give your run a descriptive name

wandb_logger = WandbLogger(
    entity=cfg.wandb_entity,  # set wandb_entity in the config; null = personal account
    project=project_name,
    name=run_name,
    log_model=False,
    save_dir="./wandb/wandb_tmp",
)

csv_logger = CSVLogger("runs", name=project_name)


## Initialize trainer

With the loggers done, now the trainer needs to be defined.  The trainer defines several properties of your training run. Here we define:

- The max number of epochs (one epoch represents your model seeing your entire training dataset).
- Define where the training run will take place (auto uses the GPU if possible, if not, CPU).
- The loggers.
- The callbacks (here we save the model with the lowest validation loss).
- Logging frequency (because we are working with a small dataset it needs to be small).


**Note that in this notebook we also set a mixed precision to reduce the model's footprint in memory.**

In [22]:
#max_epochs = 4
max_epochs = cfg.max_epochs

# -------------------------------------------------------------------------
# Trainer
# -------------------------------------------------------------------------
trainer = L.Trainer(
    max_epochs=max_epochs,
    accelerator="auto",
    devices="auto",
    precision="bf16-mixed",
    # Surya diverged outright once at lr=0.01 (losses of 10^2-10^4, exact 0.0 values under
    # bf16, val accuracy pinned at 0.5). Clipping costs nothing when gradients are small and
    # stops one bad batch taking the run with it. 4_finetune_wave_1D.py sets the same value.
    gradient_clip_val=1.0,
    logger=[wandb_logger, csv_logger],
    callbacks=[
        ModelCheckpoint(
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        )
    ],
    log_every_n_steps=2,
)

Using bfloat16 Automatic Mixed Precision (AMP)


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


## Fit the model

Finally we fit the model.  We pass the Lighting module, and our dataloaders.

In [ ]:
trainer.fit(lit_model, train_data_loader, val_data_loader)

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


wandb: Currently logged in as: lloverasdiego (surya-ws2) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


/home/jovyan/envs/surya_WS/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name  | Type      | Params | Mode  | FLOPs
----------------------------------------------------
0 | model | PeftModel | 361 M  | train | 0    
----------------------------------------------------
2.7 M     Trainable params
359 M     Non-trainable params
361 M     Total params
1,447.902 Total estimated model params size (MB)
380       Modules in train mode
0         Modules in eval mode
0         Total Flops


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

## Close the WandB run

`wandb.init()` (which `WandbLogger` calls for you) attaches a run to this *process*, not to this cell. Re-running the training cells without finishing it appends to the same run, so two experiments' curves land on top of each other and the run's summary reflects whichever finished last. Finishing it explicitly means the next run in this kernel starts clean.


In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()
    print("wandb run closed")
else:
    print("no active wandb run")


## Conclusion

With this we have now integrated our dataset, dataloaders, metrics, and DS into an end-2-end training loop and we are ready to experiment!